# DataVortex — analysis walkthrough

**Track 1 · UPI Fraud Ring & Merchant Analytics**

This notebook walks through the evidence behind the dashboard: what arrived, what the
cleaning pipeline changed, two statistical tests that decide how rankings may be read,
the hypothesis register, and the forecast backtest.

It reads the Parquet tables written by `scripts/run_pipeline.py` and
`scripts/build_analytics.py` and calls the same modules the dashboard uses. Nothing
here re-implements a metric. Rebuild it with `python scripts/build_notebook.py`.

1. Data audit
2. Cleaning evidence
3. Statistical analysis A: do merchant categories differ in dispute rate?
4. Statistical analysis B: do individual merchants differ?
5. Hypothesis register
6. Forecast validation
7. Key findings and limitations

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():          # opened from inside notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.analytics import query_engine as QE
from src.analytics.shrinkage import MIN_DENOMINATOR, estimate_prior_strength
from src.analytics.significance import homogeneity_test
from src.config import PROCESSED_DIR

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

star = {p.stem: pd.read_parquet(p) for p in PROCESSED_DIR.glob("*.parquet")}
agg = {p.stem: pd.read_parquet(p) for p in (PROCESSED_DIR / "analytics").glob("*.parquet")}
print(f"{len(star)} star-schema tables and {len(agg)} analytics tables loaded")

12 star-schema tables and 25 analytics tables loaded


## 1. Data audit

### Row counts, raw to clean

Only byte-identical duplicate rows are removed. KYC and merchant records reduce to one row
per key only after every candidate row is kept in the identity bridge table.

In [2]:
star["recon_rows"]

,Metric,Raw,Clean,Difference,Explanation
0,Transaction rows,20400,20000,-400,byte-identical duplicate rows collapsed
1,Unique transaction IDs,20000,20000,0,"unchanged — duplicates shared an ID, they did not add one"
2,Chargeback rows,2884,2800,-84,byte-identical duplicate rows collapsed
3,Unique complaint IDs,2800,2800,0,unchanged
4,KYC records,36400,28920,-7480,"exact duplicates collapsed, then resolved to one row per user_key; every candidate pre..."
5,Merchant records,6210,4343,-1867,"exact duplicates collapsed, then resolved to one row per merchant_key; every candidate..."


### Are the documented joins real?

Two ID columns drawn independently from the same ID space still overlap a lot by chance.
The test compares observed overlap with the overlap expected under independence. A z-score
near zero means the join cannot be told apart from random.

In [3]:
star["recon_independence"][["pair", "expected_overlap", "observed_overlap", "z_score", "verdict"]]

,pair,expected_overlap,observed_overlap,z_score,verdict
0,tx.user x kyc.user,5745.0,5799,0.87,consistent with chance
1,tx.merchant x mer.merchant,3885.0,3893,0.18,consistent with chance
2,cb.txn x tx.txn,1.0,2451,3420.33,genuine relationship


### Forensic facts measured by the pipeline

These are the values the dashboard quotes in its text. No page types them in by hand.

In [4]:
star["audit_facts"][["fact", "value", "unit", "description"]]

,fact,value,unit,description
0,kyc_ids_with_multiple_pans,4422.00,customer IDs,Customer IDs carrying more than one distinct valid PAN
1,pans_shared_across_ids,0.00,PANs,Valid PANs appearing under more than one customer ID
2,full_aadhaar_shared_across_ids,0.00,Aadhaar numbers,Full 12-digit Aadhaar numbers under more than one customer ID
3,masked_aadhaar_shared_across_ids,319.00,masked values,Masked Aadhaar last-4 values shared by more than one customer ID
4,masked_aadhaar_expected_by_chance,326.50,masked values,Expected shared last-4 values if customers were assigned at random
5,full_settlement_shared_across_merchants,0.00,accounts,Full settlement account numbers under more than one merchant ID
6,masked_settlement_shared_across_merchants,71.00,masked values,Masked settlement last-4 values shared by more than one merchant ID
7,masked_settlement_expected_by_chance,69.90,masked values,Expected shared last-4 values if merchants were assigned at random
8,date_slash_first_gt12,18385.00,rows,Slash dates whose first part exceeds 12 (proves DD/MM order)
9,date_slash_second_gt12,0.00,rows,Slash dates whose second part exceeds 12 (would contradict DD/MM)


## 2. Cleaning evidence

Damaged values are repaired where the repair is unambiguous and flagged either way. The
flags stay on the rows, so every exclusion in an analysis can be counted.

In [5]:
def flag_summary(name, frame):
    flags = frame.select_dtypes(include=["bool", "boolean"])
    return pd.DataFrame({
        "table": name,
        "flag": flags.columns,
        "rows_flagged": flags.sum().astype(int).values,
        "pct_of_rows": (100 * flags.mean()).astype(float).round(2).values,
    })

tx, cb = star["fact_transactions"], star["fact_chargebacks"]
pd.concat([flag_summary("fact_transactions", tx), flag_summary("fact_chargebacks", cb)],
          ignore_index=True)

,table,flag,rows_flagged,pct_of_rows
0,fact_transactions,timestamp_invalid,0,0.00
1,fact_transactions,timestamp_time_known,19000,95.00
2,fact_transactions,user_unresolved,13522,67.61
3,fact_transactions,merchant_unresolved,10369,51.84
4,fact_transactions,amount_sign_invalid,420,2.10
5,fact_transactions,amount_missing,0,0.00
6,fact_transactions,utr_valid,19000,95.00
7,fact_transactions,utr_missing,1000,5.00
8,fact_transactions,mcc_unreliable,20000,100.00
9,fact_chargebacks,txn_unlinked,193,6.89


In [6]:
agg["agg_data_quality"]

,metric,value,treatment,pct
0,Transactions (cleaned),20000,,100.00
1,Amount sign-corrupted,420,repaired to magnitude + flagged,2.10
2,UTR missing,1000,never fabricated,5.00
3,Timestamp date-only,1000,excluded from hourly analysis,5.00
4,Merchant unresolved,10369,"routed to UNKNOWN, retained",51.84
5,User unresolved,13522,"routed to UNKNOWN, retained",67.61
6,Complaints (cleaned),2800,,100.00
7,Complaints unlinked,193,no attributable transaction,6.89
8,Complaint delay unknown,412,timestamp unparseable,14.71
9,Complaint delay negative,92,"flagged, not clipped",3.29


In [7]:
users, merchants = star["dim_users"], star["dim_merchants"]
bridge = star["bridge_identity_collision"]
print(f"customer IDs shared by different people:      {int(users['identity_ambiguous'].sum()):,}")
print(f"merchant IDs shared by different businesses:  {int(merchants['identity_ambiguous'].sum()):,}")
print(f"candidate rows preserved in the bridge table: {len(bridge):,}")

customer IDs shared by different people:      5,341
merchant IDs shared by different businesses:  1,310
candidate rows preserved in the bridge table: 16,578


## 3. Statistical analysis A — do merchant categories differ in dispute rate?

A dashboard can always sort categories by rate. The question is whether the ordering is
real. A chi-square test of homogeneity compares every category's disputed-transaction
count with what one shared rate would produce. The UNKNOWN group (transactions with no
merchant master record) stays in the test, as it does on the dashboard.

In [8]:
cat = agg["agg_category"]
category_test = homogeneity_test(cat["disputed_transactions"], cat["transactions"])
display(cat[["category", "transactions", "disputed_transactions", "rate_pct",
             "ci_low_pct", "ci_high_pct", "p_adjusted", "significant"]])
category_test

,category,transactions,disputed_transactions,rate_pct,ci_low_pct,ci_high_pct,p_adjusted,significant
0,Transport,1078,144,13.358,11.457,15.520,0.944690,False
1,Telecom,972,126,12.963,10.996,15.221,0.944690,False
2,Retail,899,114,12.681,10.663,15.016,0.944690,False
3,Books & Stationery,915,116,12.678,10.677,14.991,0.944690,False
4,Hotel,925,117,12.649,10.660,14.946,0.944690,False
5,Grocery,913,114,12.486,10.498,14.789,0.992751,False
6,Apparel,1052,129,12.262,10.416,14.383,0.994035,False
7,Restaurant,893,109,12.206,10.219,14.517,0.994035,False
8,UNKNOWN (no master record),10369,1257,12.123,11.508,12.765,0.944690,False
9,Department Store,1056,122,11.553,9.763,13.622,0.944690,False


{'groups': 12,
 'pooled_rate': 0.12255,
 'chi_square': 5.241,
 'df': 11,
 'p_value': 0.9190143074909098,
 'significant': False,
 'note': 'group differences are within sampling noise'}

Cross-check: the semantic query engine the AI Investigator uses computes the same
numerators and denominators as the materialized aggregate.

In [9]:
frames = QE.prepare_frames(star)
engine = QE.run(QE.QuerySpec(metric="chargeback_to_transaction_ratio",
                             dimension="merchant_category"), frames).frame
check = (engine.set_index("dimension")[["numerator", "denominator"]]
               .join(cat.set_index("category")[["disputed_transactions", "transactions"]]))
assert (check["numerator"] == check["disputed_transactions"]).all()
assert (check["denominator"] == check["transactions"]).all()
print(f"query engine and aggregate agree for all {len(check)} categories")

query engine and aggregate agree for all 12 categories


## 4. Statistical analysis B — do individual merchants differ?

If merchants had different underlying dispute propensities, their observed rates would
spread out more than binomial sampling alone allows. The overdispersion statistic
(chi-square divided by degrees of freedom) sits near 1 when there is no extra spread.
Only merchants above the denominator floor are tested.

In [10]:
mer = agg["agg_merchant"]
eligible = mer[~mer["below_floor"]]
observed = estimate_prior_strength(eligible["disputed_transactions"], eligible["transactions"])
observed_ratio = observed["chi_square"] / observed["df"]
print(f"merchants with at least {MIN_DENOMINATOR} transactions: {len(eligible):,}")
print(f"chi-square / df = {observed_ratio:.4f}, p = {observed['p_value']:.3f}, "
      f"overdispersed = {observed['overdispersed']}")

merchants with at least 3 transactions: 3,451
chi-square / df = 0.9688, p = 0.904, overdispersed = False


Calibration by simulation: give every merchant the same dispute rate, keep each merchant's
real transaction count, and recompute the statistic 200 times. If the observed value sits
inside that range, merchant-level differences cannot be told apart from noise.

In [11]:
rng = np.random.default_rng(2026)
n = eligible["transactions"].to_numpy()
shared_rate = observed["mean_rate"]
simulated = []
for _ in range(200):
    draw = estimate_prior_strength(pd.Series(rng.binomial(n, shared_rate)), pd.Series(n))
    simulated.append(draw["chi_square"] / draw["df"])
simulated = np.array(simulated)
low, high = np.quantile(simulated, [0.025, 0.975])
print(f"observed chi-square / df:               {observed_ratio:.4f}")
print(f"one shared rate, middle 95% of 200 runs: {low:.4f} to {high:.4f}")
print(f"simulations at or above observed:        {(simulated >= observed_ratio).mean():.0%}")
print(f"raw rate spread {eligible['dispute_rate_raw_pct'].std():.2f} pp -> "
      f"after shrinkage {eligible['dispute_rate_shrunk_pct'].std():.2f} pp")

observed chi-square / df:               0.9688
one shared rate, middle 95% of 200 runs: 0.9647 to 1.0387
simulations at or above observed:        96%
raw rate spread 17.16 pp -> after shrinkage 0.69 pp


## 5. Hypothesis register

Each insight suggested by the dataset notes, plus the team's own checks, with the test
used and the verdict. CONTRADICTED means the data points the other way.

In [12]:
agg["agg_hypothesis_register"][["hypothesis", "source", "test", "statistic", "p_value", "verdict"]]

,hypothesis,source,test,statistic,p_value,verdict
0,Certain merchant categories have disproportionately high chargebacks.,track1_dataset_notes.txt,Chi-square homogeneity across 12 categories,"chi2=5.241, df=11",0.91901,NOT SUPPORTED
1,Some users appear repeatedly in disputes.,track1_dataset_notes.txt,Observed vs binomial-expected count of users with >=2 disputes,"observed=39, expected=33.9, z=+0.87",0.38398,NOT SUPPORTED
2,Some merchants show sudden transaction spikes followed by disputes.,track1_dataset_notes.txt,Poisson index of dispersion on daily volume + max merchant-day burst,"var/mean=0.901, max merchant-day=3",0.73588,NOT SUPPORTED
3,Missing UTRs correlate with failed or disputed transactions.,track1_dataset_notes.txt,"Chi-square, missing-UTR vs present-UTR, on dispute and on failure","dispute chi2=0.117, failure chi2=0.007",0.73033,NOT SUPPORTED
4,Unverified or rejected KYC users show higher dispute risk.,track1_dataset_notes.txt,"Chi-square homogeneity across 4 KYC statuses, plus per-status tests with Benjamini-Hoc...","chi2=7.894, df=3, min adjusted pairwise p=0.053",0.04742,BORDERLINE
5,Delayed chargeback reporting indicates account takeover or late fraud detection.,track1_dataset_notes.txt,"Welch t-test on reporting delay, account-takeover vs other reasons (directional)","t=-2.45, mean 5.45 d vs 6.62 d",0.01441,CONTRADICTED
6,Duplicate transactions inflate revenue and dispute metrics if not removed.,track1_dataset_notes.txt,Direct quantification against the raw file,"400 duplicate rows, Rs 5,186,378",NaN,QUANTIFIED
7,Individual merchants differ in how often their transactions are disputed.,DataVortex,Beta-binomial overdispersion test on merchants with >=3 transactions,"chi2/df=0.9688, N=3,451",0.90381,NOT SUPPORTED
8,Transaction volume follows an intraday pattern.,DataVortex,Chi-square goodness of fit against a uniform 24-hour distribution (date-only timestamp...,"chi2=17.5, df=23, CV=0.0310",0.78535,NOT SUPPORTED
9,A mismatch between reason code and complaint text is an anomaly signal.,DataVortex,Agreement rate against the independence expectation (one-sample z-test),"agreement=16.83%, expected=16.51%, z=+0.38, n=1,973",0.70510,NOT SUPPORTED


## 6. Forecast validation

Four simple methods are fitted on the earlier weeks and scored on a holdout they never saw.
The lowest holdout error is selected. The last column shows how often the 95% band
contained the actual holdout value.

In [13]:
agg["agg_forecast_backtest"]

,series,label,method,method_label,mae,rmse,mape_pct,holdout_band_coverage_pct,selected
0,transactions,Transactions per day,mean,flat mean,13.449,15.274,6.013,96.4,True
1,transactions,Transactions per day,naive,last value carried forward,13.714,16.102,6.032,96.4,False
2,transactions,Transactions per day,seasonal_naive,same weekday last week,14.929,18.881,6.653,96.4,False
3,transactions,Transactions per day,linear_trend,linear trend,13.475,15.352,6.067,96.4,False
4,disputed_transactions,Disputed transactions per day,mean,flat mean,4.675,5.778,17.460,85.7,False
5,disputed_transactions,Disputed transactions per day,naive,last value carried forward,9.286,10.637,30.788,60.7,False
6,disputed_transactions,Disputed transactions per day,seasonal_naive,same weekday last week,4.571,5.843,17.290,92.9,True
7,disputed_transactions,Disputed transactions per day,linear_trend,linear trend,4.630,5.731,17.572,89.3,False
8,transaction_value,Transaction value per day,mean,flat mean,179857.575,224680.219,6.313,96.4,False
9,transaction_value,Transaction value per day,naive,last value carried forward,174654.019,220919.967,6.208,96.4,True


In [14]:
agg["agg_forecast_diagnostics"][["label", "history_days", "holdout_days", "selected_method_label",
                                 "selected_mae", "mean_baseline_mae", "slope_per_day", "slope_p",
                                 "weekday_p", "holdout_band_coverage_pct"]]

,label,history_days,holdout_days,selected_method_label,selected_mae,mean_baseline_mae,slope_per_day,slope_p,weekday_p,holdout_band_coverage_pct
0,Transactions per day,90,28,flat mean,13.449,13.449,0.0016,0.9773,0.7011,96.4
1,Disputed transactions per day,90,28,same weekday last week,4.571,4.675,0.0136,0.4760,0.2410,92.9
2,Transaction value per day,90,28,last value carried forward,174654.019,179857.575,362.6765,0.6722,NaN,96.4


## 7. Key findings

Generated from the tables above, so the sentences change if the data changes.

In [15]:
register = agg["agg_hypothesis_register"]
notes = register[register["source"] == "track1_dataset_notes.txt"]
z = star["recon_independence"].set_index("pair")["z_score"]
verdicts = ", ".join(f"{count} {verdict}" for verdict, count in notes["verdict"].value_counts().items())
findings = [
    f"The transaction-to-KYC and transaction-to-merchant joins overlap no more than chance "
    f"(z = {z['tx.user x kyc.user']:.2f} and {z['tx.merchant x mer.merchant']:.2f}); only the "
    f"complaint-to-transaction join is real (z = {z['cb.txn x tx.txn']:.0f}).",
    f"{int(users['identity_ambiguous'].sum()):,} customer IDs and "
    f"{int(merchants['identity_ambiguous'].sum()):,} merchant IDs each belong to more than one entity.",
    f"Category dispute rates do not differ beyond chance (chi-square {category_test['chi_square']:.2f}, "
    f"df {category_test['df']}, p = {category_test['p_value']:.3f}).",
    f"Merchants show no dispute propensity beyond binomial noise "
    f"(chi-square/df {observed_ratio:.3f}, p = {observed['p_value']:.3f}).",
    f"Of the {len(notes)} insights suggested by the dataset notes: {verdicts}.",
]
for number, text in enumerate(findings, 1):
    print(f"{number}. {text}")

1. The transaction-to-KYC and transaction-to-merchant joins overlap no more than chance (z = 0.87 and 0.18); only the complaint-to-transaction join is real (z = 3420).
2. 5,341 customer IDs and 1,310 merchant IDs each belong to more than one entity.
3. Category dispute rates do not differ beyond chance (chi-square 5.24, df 11, p = 0.919).
4. Merchants show no dispute propensity beyond binomial noise (chi-square/df 0.969, p = 0.904).
5. Of the 7 insights suggested by the dataset notes: 4 NOT SUPPORTED, 1 BORDERLINE, 1 CONTRADICTED, 1 QUANTIFIED.


### Limitations

- The data has no fraud label, so no supervised fraud model can be trained or validated.
  The Risk Indicator Score is a transparent review-priority total, not a fraud probability.
- Two of the documented joins behave like random overlap, so any breakdown by customer or
  merchant attributes covers only the rows that resolve. Each chart states its coverage.
- The forecast rests on one quarter of daily history. It can show that a series has no
  trend or weekly pattern. It cannot capture seasonality longer than the data.
- Clustering was tried and dropped: the transaction graph is a forest with no repeated
  customer-merchant pairs, so there is no structure for a clustering method to recover.